# 02 · Experiments: the tracking system, and rebuilding the best model from its record

Every model in this project was trained through one function, `run_cv_experiment`,
and logged by `save_experiment` to a plain, git-tracked ledger:

* **`experiments/runs.csv`** — one row per run: scores (ROC-AUC primary, accuracy
  secondary), per-fold means and standard deviations, Kaggle public/private
  leaderboard scores, and provenance (git commit, dirty flag, data fingerprint,
  parameter hash).
* **`experiments/runs/{run_id}/`** — the full record: `params.json`, `metrics.json`,
  `notes.md`, the uncommitted `git_diff.patch`, an `environment.txt` snapshot of
  the lock file, and the out-of-fold (OOF) and test prediction arrays.

The design and the column schema are described in
[docs/experiment_tracking.md](docs/experiment_tracking.md).

This notebook demonstrates the system end to end on the project's **best single
model** (`lgbm-optuna-fe_v4_native-top2`): read its record from the ledger, rebuild
its features and parameters from that record alone, re-run the cross-validation,
and check the result against what was logged.

*Prerequisites: `python data/fetch_data.py`. Nothing is written to the ledger.*

## 1. Setup

`src/` holds the reusable, leakage-aware infrastructure; the notebook only wires it
together.

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedKFold

from src.cv import run_cv_experiment, save_experiment  # noqa: F401  (save shown, not called)
from src.features import build_features
from src.tracking import RUNS_DIR, hash_file, load_runs, train_parquet_for

## 2. The ledger

`load_runs()` returns the ledger sorted by OOF ROC-AUC. A few of its 33 columns:

In [2]:
runs = load_runs()
cols = ['run_id', 'tag', 'model_class', 'data_version', 'oof_roc_auc',
        'fold_roc_auc_std', 'lb_public', 'lb_private', 'kaggle_run']
print(f'{len(runs)} runs logged')
runs[cols].head(8).style.hide(axis='index').format(
    {'oof_roc_auc': '{:.5f}', 'fold_roc_auc_std': '{:.5f}',
     'lb_public': '{:.5f}', 'lb_private': '{:.5f}'})

72 runs logged


run_id,tag,model_class,data_version,oof_roc_auc,fold_roc_auc_std,lb_public,lb_private,kaggle_run
20260630-002812-8b56cf,stack-logit-lr-best-v2,LogitStackLR,blend_v1,0.91729,0.00097,0.91438,0.91564,False
20260628-023855-028a55,stack-logit-lr-best-v1,LogitStackLR,blend_v1,0.91722,0.00097,0.91445,0.91567,False
20260630-002742-4c9eb9,stack-logit-lr-curated-c1-v2,LogitStackLR,blend_v1,0.91719,0.00095,0.91456,0.91582,False
20260630-002846-c869ad,blend-hillclimb-curated-v2,HillClimbBlend,blend_v1,0.91716,0.00094,0.91450,0.91576,False
20260628-023826-4ad139,stack-logit-lr-curated-c1-v1,LogitStackLR,blend_v1,0.91710,0.00097,0.91451,0.91571,False
20260630-002917-9c8d1d,blend-softmax-curated-v2,SoftmaxAUCBlend,blend_v1,0.91710,0.00094,0.91440,0.91567,False
20260630-003009-224ce0,stack-lgbm-optuna-curated-v2,LGBMStack,blend_v1,0.91708,0.00095,0.91452,0.91570,False
20260628-023924-b15063,blend-hillclimb-curated-v1,HillClimbBlend,blend_v1,0.91706,0.00096,0.91444,0.91564,False


## 3. Rebuild the best single model from its record

### 3.1 What the ledger says about it

In [3]:
TAG = 'lgbm-optuna-fe_v4_native-top2'
record = runs.loc[runs['tag'] == TAG].iloc[0]
run_dir = RUNS_DIR / record['run_id']
params = json.loads((run_dir / 'params.json').read_text())

print(f"run_id        {record['run_id']}")
print(f"data_version  {record['data_version']}")
print(f"data_hash     {record['data_hash']}")
print(f"git_hash      {record['git_hash']}  (dirty={record['git_dirty']})")
print(f"OOF ROC-AUC   {record['oof_roc_auc']:.6f}   fold std {record['fold_roc_auc_std']:.6f}")
print(f"private LB    {record['lb_private']:.5f}\n")
print((run_dir / 'notes.md').read_text())

run_id        20260628-183230-86782f
data_version  fe_v4_native
data_hash     8ed62b1c00a2
git_hash      f7d3111  (dirty=True)
OOF ROC-AUC   0.916810   fold std 0.000929
private LB    0.91558

LGBM on fe_v4_native, finalized LOCALLY (CPU). Params from the narrow-search Optuna study (3-fold inner CV, ROC-AUC; multivariate TPE + MedianPruner; early stopping decides tree count; SQLite-persisted; 150 trials), warm-started from run 20260610-183508-48286b; partial study from Kaggle resumed/finalized locally. Final params chosen among top-3 trials by OUTER 5-fold OOF ROC-AUC. Notebook: kaggle/predict-customer-churn-optuna-lgbm-cpu-fe_v4.ipynb. Candidate 2/3 by inner-CV (trial 97, inner 0.916629).


### 3.2 Rebuild the features

`build_features('fe_v4_native')` from [src/features.py](src/features.py) produces
the feature set the run was trained on: the 19 raw features with the string
columns as pandas `category` dtype, plus three engineered ones — `AverageMonthly`
(`TotalCharges / tenure`) and two categorical crosses, `Contract × PaymentMethod`
and `Contract × InternetService`. The features are row-wise (stateless), so
computing them before the split cannot leak across folds.

The data fingerprint of the rebuilt training file is compared with the
`data_hash` in the ledger: equal hashes mean byte-identical training data.

In [4]:
train_df, test_df = build_features(record['data_version'])
features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X, y, X_test = train_df[features], train_df['Churn'], test_df[features]

rebuilt_hash = hash_file(train_parquet_for(record['data_version']))
print(f'{len(features)} features: {features[-3:]} added to the raw 19')
print(f"data_hash: ledger {record['data_hash']}   rebuilt {rebuilt_hash}   "
      f"match={rebuilt_hash == record['data_hash']}")

22 features: ['AverageMonthly', 'contract_x_payment', 'contract_x_internet'] added to the raw 19
data_hash: ledger 8ed62b1c00a2   rebuilt 8ed62b1c00a2   match=True


### 3.3 Re-run the cross-validation

`run_cv_experiment` runs the fold loop and returns an inspectable result without
writing anything; `save_experiment(result)` would log it. The splitter is the
project-wide `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`, the same
folds every model used, which is what makes OOF predictions from different models
comparable and stackable.

In [5]:
run_config = {
    'model_factory': lambda p: LGBMClassifier(**p),
    'params':        params,
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           f'{TAG}-rebuild',
    'notes':         f"Rebuild of {record['run_id']} from its ledger record.",
    'parent_run_id': record['run_id'],
    'save_models':   False,
    'data_version':  record['data_version'],
}
result = run_cv_experiment(run_config, X, y, X_test, features)

Run ID: 20260922-131911-362b1f
Tag:    lgbm-optuna-fe_v4_native-top2-rebuild



Fold 0: roc_auc=0.9165  accuracy=0.8617  (fit 52.4s)


Fold 1: roc_auc=0.9175  accuracy=0.8619  (fit 51.3s)


Fold 2: roc_auc=0.9169  accuracy=0.8621  (fit 51.7s)


Fold 3: roc_auc=0.9180  accuracy=0.8631  (fit 51.4s)


Fold 4: roc_auc=0.9152  accuracy=0.8618  (fit 51.5s)



OOF ROC-AUC:  0.9168   (primary)
OOF accuracy: 0.8621   (secondary)
Folds ROC-AUC: 0.9168 ± 0.0009

Run complete. Call save_experiment(result) to log this run permanently.


### 3.4 Compare with the logged run

In [6]:
logged_oof = np.load(run_dir / 'oof_proba.npy')
rebuilt_oof = result['artifacts']['oof_proba']
diff = result['oof_roc_auc'] - record['oof_roc_auc']

print(f"OOF ROC-AUC  logged {record['oof_roc_auc']:.6f}   rebuilt {result['oof_roc_auc']:.6f}   "
      f"difference {diff:+.2e}")
print(f"fold-to-fold std of the logged run: {record['fold_roc_auc_std']:.6f}")
print(f"max |rebuilt - logged| OOF probability: {np.abs(rebuilt_oof - logged_oof).max():.2e}")
print(f"Spearman(rebuilt OOF, logged OOF): {spearmanr(rebuilt_oof, logged_oof).statistic:.6f}")

OOF ROC-AUC  logged 0.916810   rebuilt 0.916810   difference -1.11e-16
fold-to-fold std of the logged run: 0.000929
max |rebuilt - logged| OOF probability: 0.00e+00


Spearman(rebuilt OOF, logged OOF): 1.000000

The rebuild is **exact**: the training data hashes to the value recorded at the time,
and the re-run reproduces every one of the 594,194 logged OOF probabilities (largest
difference 0), hence the same OOF ROC-AUC of 0.916810. The ledger row, its
`params.json` and the feature code are enough to regenerate the model without
guesswork. (LightGBM's bagging and feature-subsampling seeds have fixed defaults,
which is why a run logged with `random_state=None` is still deterministic on the
same data, library version and thread count.)

The feature importances below show where that model's signal comes from: the two
engineered `Contract` crosses carry about half of the total gain.

## 4. Other things the record supports

* **Feature importance** — LightGBM runs store per-fold-averaged gain and split
  importance:

In [7]:
fi = result['artifacts']['feature_importance'].sort_values('gain', ascending=False)
fi.assign(gain_share=fi['gain'] / fi['gain'].sum()).head(10).style.hide(axis='index').format(
    {'gain': '{:,.0f}', 'split': '{:,.0f}', 'gain_share': '{:.1%}'})

feature,gain,split,gain_share
contract_x_payment,"1,192,594","3,099",30.2%
contract_x_internet,"864,808","1,388",21.9%
tenure,"363,416","7,069",9.2%
Contract,"361,711",122,9.2%
InternetService,"213,896",148,5.4%
OnlineSecurity,"201,466",246,5.1%
TotalCharges,"190,990","10,969",4.8%
AverageMonthly,"134,457","8,966",3.4%
PaymentMethod,"97,759",329,2.5%
MonthlyCharges,"89,752","9,236",2.3%


* **Retracting a run** — `delete_run(run_id)` removes the ledger row and the
  artifact directory together. The ledger is otherwise append-only: runs that did
  not help, including the level-2 stacking experiments summarised below, stay in
  it, because what failed is part of the record.

## 5. Where the other runs came from

The ledger holds 72 runs from two environments. CPU models (gradient-boosted
trees, EBM, random forests, logistic regression) ran locally through
`run_cv_experiment`; GPU models (TabPFN, TabICL, TabM, RealMLP, GPU CatBoost) ran
on Kaggle kernels that clone this repository, regenerate the data on-platform, and
ship their artifacts back to be appended to the ledger (flagged `kaggle_run`).
`scripts/check_oof_alignment.py` proves that every saved OOF vector, from either
environment, lines up row by row with the local training data.

Hyperparameters for the gradient-boosted models came from Optuna studies in
[src/tuning.py](src/tuning.py): early stopping instead of a tuned tree count,
per-fold median pruning, and a final choice among the top trials made on the
**outer** 5-fold CV rather than the inner-CV optimum.

**Level-2 stacking — a documented negative result.** One line of work fed other
models' OOF predictions back in as features for a second-level GBDT, with and
without pseudo-labelled test rows. None of the fourteen variants beat the
single-model base, out-of-fold or on the leaderboard: the best of the ten that were
submitted reached 0.91443 private, against 0.91558 for the base model. The mechanisms are analysed in
[docs/stacking_experiments.md](docs/stacking_experiments.md); the drivers are in
[scripts/archive/](scripts/archive/). A *linear* combiner over the same OOF
vectors does help — that is `03_Blending.ipynb`.